# GraphRetro 环境配置教程

## 概述

**GraphRetro** 是 Somnath et al. (NeurIPS 2021) 提出的**半模板（Semi-Template）逆合成预测模型**，其核心思想是将逆合成分解为两个阶段：

1. **键编辑预测（Edit Prediction）**：在产物分子图上预测哪些键需要断裂或修改，生成中间体（synthons）
2. **合成子补全（Synthon Completion）**：为每个中间体添加合适的离去基团（Leaving Groups），得到完整反应物

这种两阶段的设计相比纯模板方法更灵活，相比纯无模板方法又引入了化学先验知识。

> **论文**：Somnath, V. R., Bunne, C., Coley, C. W., Krause, A., & Barzilay, R. (2021).
> *Learning Graph Models for Retrosynthesis Prediction.* NeurIPS 2021.
> **代码**：https://github.com/vsomnath/graphretro

### 本系列教程包含三个 Notebook

| 编号 | 内容 | 目标 |
|------|------|------|
| **1_环境配置** | 安装依赖、验证环境、源码结构浏览 | 搭建可运行的开发环境 |
| **2_数据处理** | SMILES标准化、反应解析、键编辑提取、图特征构建 | 理解完整数据管线 |
| **3_模型展示** | 分子图编码、键编辑预测、离去基预测、推理流程 | 理解模型架构与推理 |

## 1. 核心依赖

GraphRetro 的原始环境基于较旧版本的 Python 和 PyTorch，本教程对其进行了版本升级：

| 依赖 | 原始版本 | 教程版本 | 说明 |
|------|---------|---------|------|
| Python | 3.7.3 | **3.10+** | 更好的类型提示支持 |
| PyTorch | 1.7.0 | **2.0+** (CPU) | 更快的计算速度 |
| RDKit | 旧版 | **2023.09+** | 化学信息工具箱 |
| NetworkX | 未指定 | **3.0+** | 图/网络操作 |
| NumPy | 未指定 | **1.24+** | 数值计算 |
| joblib | 未指定 | **1.3+** | 并行处理 |
| tqdm | 未指定 | **4.66+** | 进度条 |

## 2. 虚拟环境创建

我们将在项目根目录下的 `envs/graphretro_tutorial_envs/` 中创建独立的 conda 环境，避免与其他项目冲突。

In [ ]:
import os, sys

def find_project_root(marker='source_repos'):
    """从当前目录向上查找项目根目录（包含 source_repos 的目录）"""
    path = os.path.abspath('.')
    while path != '/':
        if os.path.isdir(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f'找不到包含 {marker} 的项目根目录')

PROJECT_ROOT = find_project_root()
print(f'项目根目录: {PROJECT_ROOT}')

# 关键路径（使用相对路径）
ENV_DIR = os.path.join(PROJECT_ROOT, 'envs', 'graphretro_tutorial_envs')
SOURCE_DIR = os.path.join(PROJECT_ROOT, 'source_repos', 'graphretro')
TUTORIAL_DIR = os.path.join(PROJECT_ROOT, 'teaching_demos', '2.single_step_retro_tutorial',
                            '2.3.semi-template', '2.3.2.graphretro')

print(f'环境目录: {os.path.relpath(ENV_DIR, PROJECT_ROOT)}')
print(f'源码目录: {os.path.relpath(SOURCE_DIR, PROJECT_ROOT)}')
print(f'教程目录: {os.path.relpath(TUTORIAL_DIR, PROJECT_ROOT)}')

In [ ]:
%%bash -s "$PROJECT_ROOT"
PROJECT_ROOT=$1
ENV_DIR="${PROJECT_ROOT}/envs/graphretro_tutorial_envs"

echo "========================================"
echo "GraphRetro 教程环境配置"
echo "========================================"

# 检查 conda 是否可用
if ! command -v conda &> /dev/null; then
    echo "[ERROR] conda 未安装或不在 PATH 中"
    exit 1
fi

# 检查环境是否已存在
if [ -d "$ENV_DIR" ] && [ -f "${ENV_DIR}/bin/python" ]; then
    echo "[INFO] 环境已存在: $ENV_DIR"
    echo "[INFO] 跳过创建步骤"
else
    echo "[INFO] 创建 conda 环境..."
    conda create --prefix "$ENV_DIR" python=3.10 -y -q
    
    echo "[INFO] 安装核心依赖..."
    "${ENV_DIR}/bin/pip" install --quiet \
        torch==2.1.0+cpu -f https://download.pytorch.org/whl/torch_stable.html
    
    echo "[INFO] 安装 RDKit 和其他依赖..."
    conda install --prefix "$ENV_DIR" -c conda-forge rdkit -y -q
    
    "${ENV_DIR}/bin/pip" install --quiet \
        networkx>=3.0 \
        numpy>=1.24 \
        pandas>=2.0 \
        joblib>=1.3 \
        tqdm>=4.66 \
        matplotlib>=3.7 \
        ipykernel
    
    echo "[INFO] 安装 GraphRetro 源码为开发模式..."
    cd "${PROJECT_ROOT}/source_repos/graphretro"
    "${ENV_DIR}/bin/pip" install -e . --quiet
    
    echo "[INFO] 环境创建完成!"
fi

echo ""
echo "Python 版本: $(${ENV_DIR}/bin/python --version)"
echo "环境路径: $ENV_DIR"

## 3. 注册 Jupyter Kernel（可选）

如果需要在 Jupyter 中使用此环境，可以注册为独立的 kernel：

In [ ]:
%%bash -s "$ENV_DIR"
ENV_DIR=$1
"${ENV_DIR}/bin/python" -m ipykernel install --user --name graphretro_tutorial --display-name "GraphRetro Tutorial"
echo "[INFO] Kernel 注册完成: GraphRetro Tutorial"

## 4. 环境验证

逐一验证所有核心依赖是否正确安装。

In [ ]:
import importlib

# ============================================================
# 核心依赖检查
# ============================================================
checks = [
    ('Python',    lambda: f'{sys.version}'),
    ('PyTorch',   lambda: __import__('torch').__version__),
    ('RDKit',     lambda: __import__('rdkit').__version__),
    ('NumPy',     lambda: __import__('numpy').__version__),
    ('NetworkX',  lambda: __import__('networkx').__version__),
    ('pandas',    lambda: __import__('pandas').__version__),
    ('joblib',    lambda: __import__('joblib').__version__),
    ('tqdm',      lambda: __import__('tqdm').__version__),
    ('matplotlib',lambda: __import__('matplotlib').__version__),
]

print('=' * 60)
print(f'{"依赖":15} {"版本":20} {"状态":10}')
print('=' * 60)
all_ok = True
for name, get_ver in checks:
    try:
        ver = get_ver()
        status = '✅'
    except Exception as e:
        ver = str(e)
        status = '❌'
        all_ok = False
    print(f'{name:15} {ver:20} {status}')

print('=' * 60)
print('环境状态:', '✅ 全部就绪' if all_ok else '❌ 部分依赖缺失')

## 5. RDKit 基础功能验证

GraphRetro 深度依赖 RDKit 进行分子操作。下面验证关键功能：

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw, AllChem
import numpy as np

# ============================================================
# 5.1 SMILES 解析与标准化
# ============================================================
smi = 'c1ccc(CC(=O)O)cc1'  # 苯乙酸
mol = Chem.MolFromSmiles(smi)
canonical = Chem.MolToSmiles(mol)
print(f'原始 SMILES:  {smi}')
print(f'标准 SMILES:  {canonical}')
print(f'原子数:       {mol.GetNumAtoms()}')
print(f'键数:         {mol.GetNumBonds()}')
print()

# ============================================================
# 5.2 Kekulize 操作（GraphRetro 中的关键步骤）
# ============================================================
mol_k = Chem.MolFromSmiles(smi)
Chem.Kekulize(mol_k)
print('Kekulize 后的 SMILES:', Chem.MolToSmiles(mol_k, kekuleSmiles=True))
print('说明: Kekulize 将芳香键转换为交替的单键/双键表示')
print()

# ============================================================
# 5.3 原子属性提取（GraphRetro 特征基础）
# ============================================================
print('分子原子属性表：')
print(f'{"Idx":>4} {"Symbol":>6} {"Degree":>6} {"FCharge":>7} {"Hybrid":>8} {"Valence":>7} {"NumHs":>5} {"Arom":>5}')
print('-' * 55)
for atom in mol.GetAtoms():
    print(f'{atom.GetIdx():4d} {atom.GetSymbol():>6} {atom.GetDegree():>6d} '
          f'{atom.GetFormalCharge():>7d} {str(atom.GetHybridization()):>8} '
          f'{atom.GetTotalValence():>7d} {atom.GetTotalNumHs():>5d} '
          f'{atom.GetIsAromatic():>5}')
print()

# ============================================================
# 5.4 键属性提取
# ============================================================
print('分子键属性表：')
print(f'{"Idx":>4} {"Begin":>5} {"End":>5} {"Type":>10} {"Conj":>5} {"Ring":>5}')
print('-' * 40)
for bond in mol.GetBonds():
    print(f'{bond.GetIdx():4d} {bond.GetBeginAtomIdx():>5d} {bond.GetEndAtomIdx():>5d} '
          f'{str(bond.GetBondType()):>10} {bond.GetIsConjugated():>5} {bond.IsInRing():>5}')

## 6. PyTorch 与图操作验证

GraphRetro 使用 PyTorch 实现消息传递神经网络（MPNN），下面验证基本的张量操作：

In [ ]:
import torch
import torch.nn as nn
import networkx as nx

# ============================================================
# 6.1 GRU 基础操作（GraphRetro 用 GRU 进行消息传递）
# ============================================================
gru = nn.GRU(input_size=10, hidden_size=20, num_layers=1, batch_first=True)
x = torch.randn(1, 5, 10)  # (batch, seq_len, features)
output, h_n = gru(x)
print(f'GRU 输入形状:  {x.shape}')
print(f'GRU 输出形状:  {output.shape}')
print(f'GRU 隐状态形状: {h_n.shape}')
print()

# ============================================================
# 6.2 index_select_ND 模拟（GraphRetro 核心操作）
# ============================================================
# 这是消息传递中的关键操作：根据邻居索引选取特征
def index_select_ND(source, dim, index):
    """按 index 从 source 的 dim 维度选取，支持高维索引"""
    index_size = index.size()
    suffix_dim = source.size()[1:]
    final_size = index_size + suffix_dim
    target = source.index_select(dim, index.view(-1))
    return target.view(final_size)

# 模拟 5 个原子，每个 4 维特征，邻接表
atom_features = torch.randn(5, 4)
neighbors = torch.tensor([[1, 2, 0], [0, 3, 0], [0, 4, 0], [1, 0, 0], [2, 0, 0]])  # (5, 3)
nei_features = index_select_ND(atom_features, 0, neighbors)
print(f'原子特征形状: {atom_features.shape}')
print(f'邻接表形状:   {neighbors.shape}')
print(f'邻居特征形状: {nei_features.shape}  # (原子数, 邻居数, 特征维)')
print()

# ============================================================
# 6.3 NetworkX 图操作（GraphRetro 用于分子图表示）
# ============================================================
# 构建一个简单的分子图
G = nx.Graph()
G.add_nodes_from([(0, {'label': 'C'}), (1, {'label': 'C'}), (2, {'label': 'O'})])
G.add_edges_from([(0, 1, {'label': 1}), (1, 2, {'label': 2})])

print('NetworkX 图属性:')
print(f'  节点数: {G.number_of_nodes()}')
print(f'  边数:   {G.number_of_edges()}')
print(f'  邻接矩阵:\n{nx.to_numpy_array(G)}')
print(f'  连通分量数: {nx.number_connected_components(G)}')

## 7. GraphRetro 源代码结构

下面展示 GraphRetro 代码仓库的核心模块结构：

In [ ]:
import os

def print_tree(path, prefix='', max_depth=3, current_depth=0, exclude_dirs=None):
    """打印目录树结构"""
    if exclude_dirs is None:
        exclude_dirs = {'__pycache__', '.git', '*.egg-info', 'datasets'}
    if current_depth >= max_depth:
        return

    entries = sorted(os.listdir(path))
    dirs = [e for e in entries if os.path.isdir(os.path.join(path, e)) and e not in exclude_dirs 
            and not e.endswith('.egg-info')]
    files = [e for e in entries if os.path.isfile(os.path.join(path, e))]

    for f in files:
        print(f'{prefix}├── {f}')
    for i, d in enumerate(dirs):
        connector = '└── ' if i == len(dirs) - 1 else '├── '
        print(f'{prefix}{connector}{d}/')
        extension = '    ' if i == len(dirs) - 1 else '│   '
        print_tree(os.path.join(path, d), prefix + extension, max_depth, current_depth + 1, exclude_dirs)

print('GraphRetro 源码结构：')
print('graphretro/')
print_tree(SOURCE_DIR, max_depth=3)

## 8. GraphRetro 源码模块概览

GraphRetro 的核心模块分布在 `seq_graph_retro/` 包中，主要包括以下子模块：

| 模块 | 路径 | 功能 |
|------|------|------|
| **molgraph** | `seq_graph_retro/molgraph/` | 分子图表示与特征提取 |
| **layers** | `seq_graph_retro/layers/` | GNN 编码器层（MPNN, WLN, GTransformer） |
| **models** | `seq_graph_retro/models/` | 完整模型（SingleEdit, LGIndEmbed） |
| **data** | `seq_graph_retro/data/` | 数据集类和批处理函数 |
| **utils** | `seq_graph_retro/utils/` | 反应解析、化学操作、评估工具 |
| **search** | `seq_graph_retro/search/` | 束搜索推理 |

### 数据预处理脚本

| 脚本 | 功能 |
|------|------|
| `data_process/canonicalize_prod.py` | 产物 SMILES 标准化与原子重映射 |
| `data_process/parse_info.py` | 反应信息解析（提取编辑和离去基） |
| `data_process/core_edits/bond_edits.py` | 键编辑训练数据生成 |
| `data_process/lg_edits/lg_classifier.py` | 离去基分类训练数据生成 |

## 9. GraphRetro 完整工作流程

### 训练阶段

```
原始反应 SMILES
    │
    ▼
产物标准化 (canonicalize_prod.py)
    │
    ▼
反应信息解析 (parse_info.py)
    ├── 键编辑 (core_edits): "a1:a2:old_bo:new_bo"
    ├── 离去基 (lg_edits): 新增的键和原子组
    └── 附着原子 (attach_atoms): 离去基的连接点
    │
    ├──────────────────────┐
    ▼                      ▼
键编辑数据 (bond_edits.py)  离去基数据 (lg_classifier.py)
    │                      │
    ▼                      ▼
SingleEdit 模型训练     LGIndEmbed 模型训练
```

### 推理阶段

```
产物 SMILES p
    │
    ▼
① SingleEdit: 预测键编辑 → P(edit | p)
    │
    ▼
② 应用编辑: 断键/改键 → 生成中间体 (synthons)
    │
    ▼
③ LGIndEmbed: 预测离去基 → P(lg | synthon, p)
    │
    ▼
④ 组装: synthon + leaving group → 反应物
```

### 核心数学表达

GraphRetro 的预测目标可以表示为：

$$P(\mathbf{r} | \mathbf{p}) = P(\text{edits} | \mathbf{p}) \cdot \prod_{i=1}^{K} P(\text{lg}_i | \mathbf{s}_i, \mathbf{p})$$

其中 $\mathbf{p}$ 是产物，$\mathbf{r}$ 是反应物，$\mathbf{s}_i$ 是第 $i$ 个合成子（synthon），$K$ 是合成子数量。

## 10. GraphRetro 源码导入验证

验证 GraphRetro 的源码是否可以正确导入：

In [ ]:
# 确保源码可以导入
sys.path.insert(0, SOURCE_DIR)

try:
    from seq_graph_retro.molgraph.mol_features import (
        ATOM_LIST, ATOM_FDIM, BOND_FDIM, BOND_TYPES, BOND_FLOATS,
        get_atom_features, get_bond_features
    )
    print('✅ mol_features 导入成功')
    print(f'   ATOM_FDIM = {ATOM_FDIM}  (原子特征维度)')
    print(f'   BOND_FDIM = {BOND_FDIM}  (键特征维度)')
    print(f'   支持的原子类型数: {len(ATOM_LIST)}')
    print(f'   支持的键类型: {BOND_FLOATS}')
except ImportError as e:
    print(f'❌ mol_features 导入失败: {e}')

try:
    from seq_graph_retro.molgraph.rxn_graphs import RxnElement, MultiElement, BondEditsRxn
    print('✅ rxn_graphs 导入成功')
except ImportError as e:
    print(f'❌ rxn_graphs 导入失败: {e}')

try:
    from seq_graph_retro.utils.parse import get_reaction_info, get_reaction_core, ReactionInfo
    print('✅ utils.parse 导入成功')
except ImportError as e:
    print(f'❌ utils.parse 导入失败: {e}')

print()
print('GraphRetro 原子特征组成：')
print(f'  原子类型 one-hot: {len(ATOM_LIST)} 维 ({ATOM_LIST[:5]}...)')
print(f'  度数:             10 维 (0-9)')
print(f'  形式电荷:          5 维 (-2,-1,0,1,2)')
print(f'  杂化类型:          5 维 (SP,SP2,SP3,SP3D,SP3D2)')
print(f'  化合价:            7 维 (0-6)')
print(f'  氢原子数:          5 维 (0,1,3,4,5)')
print(f'  芳香性:            1 维 (bool)')
print(f'  总计: {ATOM_FDIM} 维')

## 总结

本 Notebook 完成了以下工作：

| 步骤 | 状态 |
|------|------|
| Conda 虚拟环境创建 | ✅ |
| Python / PyTorch / RDKit 安装 | ✅ |
| 核心依赖验证 | ✅ |
| RDKit 分子操作验证 | ✅ |
| PyTorch / NetworkX 操作验证 | ✅ |
| GraphRetro 源码结构浏览 | ✅ |
| GraphRetro 工作流程概览 | ✅ |

**下一步**：进入 `2_数据处理.ipynb`，学习 GraphRetro 的完整数据处理管线。